# Initial RCP-QI experiment
Run the CLI first (`uv run rcp-qi-research`). This notebook inspects the saved, seeded experiment. RCP-QI is a research hypothesis; these small noiseless simulations do not establish a speedup.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

path = Path('../results/initial.json')
data = json.loads(path.read_text(encoding='utf-8'))
data['metadata']

In [ ]:
for row in data['summary']:
    print(f"{row['case']:16} {row['algorithm']:9} success={row['success_probability_empirical']:.2f} [{row['success_ci95_low']:.2f}, {row['success_ci95_high']:.2f}] queries={row['mean_total_predicate_queries']:.2f} gates={row['mean_gate_count_total']:.0f} depth={row['mean_circuit_depth_total']:.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for algorithm in ('classical', 'grover', 'rcp_qi'):
    rows = sorted((r for r in data['summary'] if r['algorithm'] == algorithm and r['case'].startswith('equality')), key=lambda r: r['N'])
    ax.plot([r['N'] for r in rows], [r['mean_total_predicate_queries'] for r in rows], marker='o', label=algorithm)
ax.set(xlabel='N', ylabel='mean total predicate calls', title='Small-instance resource comparison')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.grid(alpha=.25)
ax.legend();

## One-iteration correct-half probability
The analytic prediction is $1/2 + 4/N - 8/N^2$. The points below are exact statevector probabilities, not sampled frequencies.

In [ ]:
one = json.loads(Path('../results/one_iteration.json').read_text(encoding='utf-8'))
rows = [r for r in one['rows'] if r['correct_half'] == 'right']
print('maximum absolute discrepancy:', max(r['absolute_error'] for r in one['rows']))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([r['N'] for r in rows], [r['analytic_probability'] for r in rows], label='analytic')
ax.scatter([r['N'] for r in rows], [r['statevector_probability'] for r in rows], label='statevector')
ax.axhline(.5, color='gray', linestyle='--', label='uninformed half')
ax.set(xlabel='N', ylabel='P(correct half)', title='One Grover iteration')
ax.set_xscale('log', base=2)
ax.legend();